In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/simulated-roads-accident-data/synthetic_road_accidents_10k.csv
/kaggle/input/simulated-roads-accident-data/synthetic_road_accidents_2k.csv
/kaggle/input/simulated-roads-accident-data/synthetic_road_accidents_100k.csv
/kaggle/input/playground-series-s5e10/sample_submission.csv
/kaggle/input/playground-series-s5e10/train.csv
/kaggle/input/playground-series-s5e10/test.csv


In [2]:
train = pd.read_csv("/kaggle/input/playground-series-s5e10/train.csv")

In [3]:
test = pd.read_csv('/kaggle/input/playground-series-s5e10/test.csv')

In [4]:
train

,id,road_type,num_lanes,curvature,speed_limit,lighting,weather,road_signs_present,public_road,time_of_day,holiday,school_season,num_reported_accidents,accident_risk
0,0,urban,2,0.06,35,daylight,rainy,False,True,afternoon,False,True,1,0.13
1,1,urban,4,0.99,35,daylight,clear,True,False,evening,True,True,0,0.35
2,2,rural,4,0.63,70,dim,clear,False,True,morning,True,False,2,0.30
3,3,highway,4,0.07,35,dim,rainy,True,True,morning,False,False,1,0.21
4,4,rural,1,0.58,60,daylight,foggy,False,False,evening,True,False,1,0.56
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
517749,517749,highway,4,0.10,70,daylight,foggy,True,True,afternoon,False,False,2,0.32
517750,517750,rural,4,0.47,35,daylight,rainy,True,True,morning,False,False,1,0.26
517751,517751,urban,4,0.62,25,daylight,foggy,False,False,afternoon,False,True,0,0.19
517752,517752,highway,3,0.63,25,night,clear,True,False,afternoon,True,True,3,0.51


In [5]:
orig_dfs = []
for k in [2, 10, 100]:
    df = pd.read_csv(f"/kaggle/input/simulated-roads-accident-data/synthetic_road_accidents_{k}k.csv")
    orig_dfs.append(df)
orig = pd.concat(orig_dfs, axis=0, ignore_index=True)

In [6]:
orig['id'] = np.arange(len(orig)) + test['id'].max() + 1
orig = orig[train.columns]
TARGET = 'accident_risk'

In [7]:
orig

,id,road_type,num_lanes,curvature,speed_limit,lighting,weather,road_signs_present,public_road,time_of_day,holiday,school_season,num_reported_accidents,accident_risk
0,690339,rural,2,0.72,60,daylight,clear,True,False,afternoon,False,False,2,0.37
1,690340,highway,4,0.95,45,daylight,foggy,False,True,evening,False,True,1,0.40
2,690341,rural,1,0.72,25,night,rainy,False,False,evening,True,False,1,0.55
3,690342,rural,4,0.86,70,dim,foggy,True,False,morning,True,True,1,0.56
4,690343,highway,1,0.00,60,night,rainy,True,True,morning,True,True,3,0.54
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
111995,802334,rural,2,0.61,60,dim,foggy,False,True,evening,False,False,1,0.54
111996,802335,rural,2,0.52,35,dim,foggy,True,True,afternoon,False,False,1,0.18
111997,802336,rural,2,0.08,70,daylight,clear,False,False,evening,True,False,1,0.20
111998,802337,rural,1,0.83,70,dim,foggy,False,True,morning,False,True,0,0.58


In [8]:
train = pd.concat([train, orig], ignore_index=True, sort=False)

In [9]:
import pandas as pd
import numpy as np
import scipy.stats
import xgboost as xgb
from xgboost import XGBRegressor
import joblib  # For saving the model and other components

# --- 1. Your Helper Functions (with Clipper class) ---

def f(X):
    """Function for feature engineering (for clipping)."""
    return \
    0.35 * X["curvature"] + \
    0.05 * int(X["lighting"] == "night") + \
    0.1 * int(X["weather"] != "clear") + \
    0.35 * int(X["speed_limit"] >= 60) + \
    0.2 * int(X["num_reported_accidents"] > 2)

class Clipper:
    """
    Class to handle the clipping logic.
    This object CAN be pickled by joblib.
    """
    def __init__(self, f_func):
        self.f_func = f_func
        self.sigma = 0.05

    def __call__(self, X):
        """Makes instances of the class callable."""
        mu = self.f_func(X)  # Apply the stored function 'f'
        sigma = self.sigma
        a, b = -mu/sigma, (1-mu)/sigma
        Phi_a, Phi_b = scipy.stats.norm.cdf(a), scipy.stats.norm.cdf(b)
        phi_a, phi_b = scipy.stats.norm.pdf(a), scipy.stats.norm.pdf(b)
        return mu*(Phi_b-Phi_a) + sigma*(phi_a-phi_b) + 1 - Phi_b

def feature_engineering_with_clip(train_df, test_df, target):
    """
    Performs all feature engineering and returns the processed dataframes,
    the clipper object, and the preprocessing artifacts dictionary.
    """
    train, test = train_df.copy(), test_df.copy()
    
    # This dictionary will store everything we need for deployment
    preprocessing_artifacts = {
        'freq_maps': {},
        'freq_means': {},
        'bin_edges': {},
        'map_num_reported': {},
        'cat_cols': [],
        'num_cols': [],
        'cols_to_remove': []
    }

    # Identify features
    cols = train.drop(columns=target).columns.tolist()
    cat = [col for col in cols if train[col].dtype in ["object", "category"] and col != target]
    preprocessing_artifacts['cat_cols'] = cat
    num = [col for col in cols if train[col].dtype not in ["object", "category", "bool"] and col not in ["id", target]]
    preprocessing_artifacts['num_cols'] = num

    # 1. Frequency Encoding
    for col in cat:
        freq = train[col].value_counts(normalize=True)
        mean_freq = train[col].map(freq).mean() 
        
        train[f"{col}_freq"] = train[col].map(freq)
        test[f"{col}_freq"] = test[col].map(freq).fillna(mean_freq)
        
        preprocessing_artifacts['freq_maps'][col] = freq
        preprocessing_artifacts['freq_means'][col] = mean_freq

    # 2. Binning Numeric Features
    for col in num:
        for q in [5, 10, 15]:
            try:
                train[f"{col}_bin{q}"], bins = pd.qcut(train[col], q=q, labels=False, retbins=True, duplicates="drop")
                test[f"{col}_bin{q}"] = pd.cut(test[col], bins=bins, labels=False, include_lowest=True)
                preprocessing_artifacts['bin_edges'][f"{col}_bin{q}"] = bins
            except Exception:
                train[f"{col}_bin{q}"] = test[f"{col}_bin{q}"] = 0
                preprocessing_artifacts['bin_edges'][f"{col}_bin{q}"] = [-np.inf, np.inf]

    # 3. Mapping specific columns
    map_col = "num_reported_accidents"
    if map_col in train.columns:
        map_num_reported = {0: 0, 1: 0, 2: 0, 3: 2, 4: 4, 5: 3, 6: 1, 7: 0}
        train[map_col] = train[map_col].map(map_num_reported)
        test[map_col] = test[map_col].map(map_num_reported)
        preprocessing_artifacts['map_num_reported'] = map_num_reported

    # 4. Drop unnecessary columns
    remove = ["time_of_day", "num_lanes", "road_type", "road_signs_present", "id_freq", "id"]
    preprocessing_artifacts['cols_to_remove'] = remove
    train.drop(columns=[col for col in remove if col in train.columns], inplace=True)
    test.drop(columns=[col for col in remove if col in test.columns], inplace=True)
    train.drop_duplicates(inplace=True)
    
    # Update cat list after drops
    cat = [col for col in cat if col in train.columns]
    preprocessing_artifacts['cat_cols'] = cat # Update artifact
    
    for col in cat:
        if col in test.columns:
            train[col] = train[col].astype("category")
            test[col] = test[col].astype("category")

    # 5. Apply clipping
    clipper = Clipper(f) 
    train["curvature_clipped"] = train.apply(clipper, axis=1)
    test["curvature_clipped"] = test.apply(clipper, axis=1)
    
    new_num = train.drop(columns=cat + [target]).columns.tolist()
    
    # Return all 5 items
    return train, test, new_num, clipper, preprocessing_artifacts

# --- 2. Load Data and Preprocess ---

# Load Data
df = train
df_test_original = pd.read_csv("/kaggle/input/playground-series-s5e10/test.csv")

# Save test IDs before they get dropped
test_ids = df_test_original["id"]

# Target column
target = df.columns.tolist()[-1]

# Apply feature engineering and capture all 5 outputs
df, df_test, new_num, clipper_object, prep_artifacts = feature_engineering_with_clip(df, df_test_original, target)

# Check the resulting dataframe
print("Processed training data head:")
print(df.head())

# Create DMatrix for CV
X_train = df.drop(columns=target)
y_train = df[target]
dtrain = xgb.DMatrix(X_train, label=y_train, enable_categorical=True)

# --- 3. Define Parameters and Run CV ---

# Define XGBoost parameters using your Optuna results
xgb_params  = {
    # Static parameters
    'tree_method': 'hist', 
    'device': 'cpu', 
    'eval_metric': 'rmse',
    'random_state': 42, 
    'max_bin': 512,
    'scale_pos_weight': 0.3615894752587659, # From your original script
    
    # Parameters from Optuna Trial 2
    'max_depth': 8,
    'learning_rate': 0.005404103854647328,
    'subsample': 0.7824279936868144,
    'colsample_bytree': 0.9140703845572055,
    'colsample_bylevel': 0.6798695128633439,
    'colsample_bynode': 0.8056937753654446,
    'gamma': 0.0005486767416600901,
    'reg_alpha': 2.3528990899815284e-08,
    'reg_lambda': 0.0007250347382396634,
    'min_child_weight': 2,
    'max_delta_step': 1
}

print("Running Cross-Validation to find best n_estimators...")
# Run cross-validation
cv_results = xgb.cv(
    params=xgb_params,
    dtrain=dtrain,
    nfold=7,
    num_boost_round=1000, # High number to allow early stopping
    metrics='rmse',
    verbose_eval=100,
    early_stopping_rounds=50 # Will stop if rmse doesn't improve
)

# Display last few CV results
print(cv_results.tail())

# Extract best boosting round
best_round = cv_results['test-rmse-mean'].idxmin()
best_rmse = cv_results['test-rmse-mean'][best_round]
print(f"Best round index: {best_round}, Best CV RMSE: {best_rmse:.7f}")

# Set n_estimators to the best round + 1
best_n_estimators = best_round + 1
xgb_params["n_estimators"] = best_n_estimators
print(f"Setting n_estimators for final model: {best_n_estimators}")

# --- 4. Final Training and Saving ---

# Final training
print("Training final model...")
model = XGBRegressor(**xgb_params, enable_categorical=True)
model.fit(X_train, y_train) # Use the full X_train/y_train

# Save all 3 model/artifact files
joblib.dump(model, "xgboost_model.pkl")
print("Model saved to xgboost_model.pkl")

joblib.dump(clipper_object, "feature_engineering_clipper.pkl")
print("Feature engineering clipper saved to feature_engineering_clipper.pkl")

joblib.dump(prep_artifacts, "preprocessing_artifacts.pkl") 
print("Preprocessing artifacts saved to preprocessing_artifacts.pkl")

# --- 5. Prediction and Submission ---

# Predict on test set
print("Predicting on test set...")
pred = model.predict(df_test)

# Prepare submission
sub = pd.DataFrame({
    "id": test_ids,
    target: pred
})

# Save submission file
sub.to_csv("submission.csv", index=False)
print("Submission file created successfully!")

Processed training data head:
   curvature  speed_limit  lighting weather  public_road  holiday  \
0       0.06           35  daylight   rainy         True    False   
1       0.99           35  daylight   clear        False     True   
2       0.63           70       dim   clear         True     True   
3       0.07           35       dim   rainy         True    False   
4       0.58           60  daylight   foggy        False     True   

   school_season  num_reported_accidents  accident_risk  road_type_freq  ...  \
0           True                     0.0           0.13        0.331587  ...   
1           True                     0.0           0.35        0.331587  ...   
2          False                     0.0           0.30        0.333141  ...   
3          False                     0.0           0.21        0.335272  ...   
4          False                     0.0           0.56        0.333141  ...   

   curvature_bin5  curvature_bin10  curvature_bin15  speed_limit_bin5  \
0